# Evaluate the Raw ensembel and develop a first Baseline

The energy score of the raw ensemble with non normalized values is [27.550657, 20.77709] for t2m and 10m wind speed.

For normalized values, the energy score is [3.3721657, 10.583179]

In [1]:
from genpp.data import FORECAST_ENS_PATH, OBSERVATIONS_PATH, FC_VARS, MISSING_DAYS
from genpp.data.utils import get_time_intersection
import xarray as xr
import dask as da  # noqa: F401
from scoringrules import es_ensemble
from einops import rearrange
import numpy as np
from tqdm import trange

In [2]:
ens = xr.open_dataset(
    FORECAST_ENS_PATH,
    chunks={
        "time": "auto",
        "number": -1,
        "latitude": -1,
        "longitude": -1,
        "level": -1,
    },
)
ens = ens[FC_VARS]

obs = xr.open_dataset(
    OBSERVATIONS_PATH,
    chunks="auto",
)
obs = obs[FC_VARS]

# Cut out the missing days first, since they are in time, not prediction_time
ens = ens.sel(time=~ens.time.isin(MISSING_DAYS))
ens = ens.assign_coords(prediction_time=ens.time + ens.prediction_timedelta).swap_dims(
    {"time": "prediction_time"}
)

times = get_time_intersection(ens, obs)

ens = ens.sel(prediction_time=times)
obs = obs.sel(time=times)
ens = ens.to_array().transpose("prediction_time", "number", "variable", "longitude", "latitude")

obs = obs.to_array()
obs = obs.transpose("time", "variable", "longitude", "latitude")

In [3]:
ens_npy = ens.values
obs_npy = obs.values

In [4]:
obs_npy: np.ndarray = rearrange(obs_npy, "t v x y -> t v (x y)")  # type: ignore
ens_npy: np.ndarray = rearrange(ens_npy, "t n v x y -> t n v (x y)")  # type: ignore

In [5]:
obs_npy.shape

(3647, 2, 1147)

In [6]:
# Process data in chunks of 100 time steps to avoid memory issues
chunk_size = 100
results = []

total_time_steps = obs_npy.shape[0]
print(f"Processing {total_time_steps} time steps in chunks of {chunk_size}")

for i in trange(0, total_time_steps, chunk_size):
    end_idx = min(i + chunk_size, total_time_steps)

    # Extract chunk
    obs_chunk = obs_npy[i:end_idx]
    ens_chunk = ens_npy[i:end_idx]

    # Compute ensemble score for this chunk
    res_chunk = es_ensemble(obs_chunk, ens_chunk, m_axis=1, v_axis=-1)
    results.append(res_chunk)

# Concatenate all results
res = np.concatenate(results, axis=0)
print(f"Final result shape: {res.shape}")
res

Processing 3647 time steps in chunks of 100


100%|██████████| 37/37 [00:43<00:00,  1.17s/it]

Final result shape: (3647, 2)


array([[21.03015 , 21.043549],
       [25.522875, 38.481678],
       [16.232574, 29.272585],
       ...,
       [22.166098, 17.220913],
       [19.835793, 21.254383],
       [24.535027, 33.35298 ]], shape=(3647, 2), dtype=float32)

In [7]:
res.mean(axis=0)

array([27.550657, 20.77709 ], dtype=float32)

## Normalized

In [8]:
obs_mean = np.mean(obs_npy, axis=0, keepdims=True)
ens_mean = np.mean(ens_npy, axis=(0, 1), keepdims=True)

obs_std = np.std(obs_npy, axis=0, keepdims=True)
ens_std = np.std(ens_npy, axis=(0, 1), keepdims=True)

obs_standardized = (obs_npy - obs_mean) / obs_std
ens_standardized = (ens_npy - ens_mean) / ens_std

In [9]:
# Process data in chunks of 100 time steps to avoid memory issues
chunk_size = 100
results = []

total_time_steps = obs_standardized.shape[0]
print(f"Processing {total_time_steps} time steps in chunks of {chunk_size}")

for i in trange(0, total_time_steps, chunk_size):
    end_idx = min(i + chunk_size, total_time_steps)

    # Extract chunk
    obs_chunk = obs_standardized[i:end_idx]
    ens_chunk = ens_standardized[i:end_idx]

    # Compute ensemble score for this chunk
    res_chunk = es_ensemble(obs_chunk, ens_chunk, m_axis=1, v_axis=-1)
    results.append(res_chunk)

# Concatenate all results
res = np.concatenate(results, axis=0)
print(f"Final result shape: {res.shape}")
res

Processing 3647 time steps in chunks of 100


100%|██████████| 37/37 [00:43<00:00,  1.17s/it]

Final result shape: (3647, 2)


array([[ 2.3382776, 11.236837 ],
       [ 3.2511346, 19.073612 ],
       [ 1.709325 , 14.992278 ],
       ...,
       [ 2.4950917,  8.521025 ],
       [ 2.321423 ,  9.540014 ],
       [ 3.0114188, 14.1451435]], shape=(3647, 2), dtype=float32)

In [10]:
res.mean(axis=0)

array([ 3.3721657, 10.583179 ], dtype=float32)

## Does the dataloader even work?
 -> It does

In [11]:
import xbatcher
from xbatcher.loaders.torch import MapDataset
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm

In [38]:
ens_gen = xbatcher.BatchGenerator(
    ens.compute(),
    input_dims={"number": 50, "variable": 2, "longitude": 37, "latitude": 31},
    batch_dims={"prediction_time": 1},
)
obs_gen = xbatcher.BatchGenerator(
    obs.compute(),
    input_dims={"variable": 2, "longitude": 37, "latitude": 31},
    batch_dims={"time": 1},
)

map_ds = MapDataset(
    ens_gen,
    obs_gen,
)

In [39]:
dl = DataLoader(map_ds, batch_size=None, num_workers=4, prefetch_factor=3, shuffle=True)

In [40]:
xs, ys = [], []
for batch in tqdm(dl):
    x, y = batch
    xs.append(x)
    ys.append(y)

xs = torch.cat(xs, dim=0).cpu().numpy()
ys = torch.cat(ys, dim=0).cpu().numpy()

  0%|          | 0/3647 [00:00<?, ?it/s]

100%|██████████| 3647/3647 [00:04<00:00, 743.35it/s]


In [41]:
xs = rearrange(xs, "b n v x y -> b n v (x y)")
ys = rearrange(ys, "b v x y -> b v (x y)")

In [42]:
# Process data in chunks of 100 time steps to avoid memory issues
chunk_size = 100
results = []

total_time_steps = xs.shape[0]
print(f"Processing {total_time_steps} time steps in chunks of {chunk_size}")

for i in trange(0, total_time_steps, chunk_size):
    end_idx = min(i + chunk_size, total_time_steps)

    # Extract chunk
    obs_chunk = ys[i:end_idx]
    ens_chunk = xs[i:end_idx]

    # Compute ensemble score for this chunk
    res_chunk = es_ensemble(obs_chunk, ens_chunk, m_axis=1, v_axis=-1)
    results.append(res_chunk)

# Concatenate all results
res = np.concat(results, axis=0)
print(f"Final result shape: {res.shape}")
res

Processing 3647 time steps in chunks of 100


  0%|          | 0/37 [00:00<?, ?it/s]

100%|██████████| 37/37 [00:43<00:00,  1.18s/it]

Final result shape: (3647, 2)


array([[20.441946, 18.424412],
       [19.979925, 28.05665 ],
       [31.172134, 14.974238],
       ...,
       [28.33645 , 16.820953],
       [22.00327 , 15.050685],
       [30.667753, 21.38252 ]], shape=(3647, 2), dtype=float32)

In [44]:
res.mean(axis=0)

array([27.550623, 20.777119], dtype=float32)